In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
SILVER_PATH_GERAL   = "workspace.case_spark_cvm.silver_cvm_fii_geral"
NOME_TABELA         = f"gold_dim_fii" 
GOLD_PATH           = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC           = int(datetime.now().strftime("%Y%m%d"))

### 1. CVM FIIs Geral

In [0]:
df_cvm_fii_geral =  spark.read.table(SILVER_PATH_GERAL)

### 2. Normaliza tipo_fundo_classe

In [0]:
df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
  "tipo_fundo_classe",
  f.when(f.col("tipo_fundo_classe").isNull(), "Fundo")
   .otherwise(f.col("tipo_fundo_classe"))
)

### 3. Criação da Janela e Prioridade

In [0]:
df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
    "prioridade_tipo",
    f.when(f.col("tipo_fundo_classe") == "Classe", 1)
     .otherwise(2)
)

In [0]:
window_mes = Window.partitionBy("cnpj_fundo_classe", "data_referencia").orderBy("prioridade_tipo")

df_cvm_fii_geral = df_cvm_fii_geral.withColumn(
    "rn_mes",
    f.row_number().over(window_mes)
).filter(f.col("rn_mes") == 1)

In [0]:
window_ultimo = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("data_referencia").desc())

df_cvm_fii_geral = df_cvm_fii_geral\
    .withColumn("rn_ultimo", f.row_number().over(window_ultimo))\
    .filter(f.col("rn_ultimo") == 1)\
    .select(
        "cnpj_fundo_classe",
        "nome_fundo_classe",
        "tipo_fundo_classe",
        "codigo_isin",
        "mandato",
        "segmento_atuacao",
        "tipo_gestao",
        "publico_alvo",
        "prazo_duracao",
        "mercado_negociacao_bolsa",
        "mercado_negociacao_mb",
        "fundo_exclusivo",
        "cotistas_vinculo_familiar",
        "nome_administrador",
        "cnpj_administrador",
        "data_funcionamento",
        "cidade",
        "estado"
    )

### 4. Salvando o Cubo

In [0]:
df_cvm_fii_geral.display()

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}")

PipelineConfig.gravar_dimensao_gold(
    spark=spark,
    df_novo=df_cvm_fii_geral,
    tabela_destino=GOLD_PATH,
    chave_pk=['cnpj_fundo_classe']

)

log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
%sql
select
    *
from workspace.case_spark_cvm.gold_dim_fii